# Injury Detection Model
## Robot Dog Search & Rescue CV Pipeline

This notebook builds a model that classifies detected people as **OK** (standing/upright) or **POTENTIALLY INJURED** (lying down) using YOLOv8-pose keypoints + a custom trained classifier.

### Pipeline
1. YOLOv8-pose detects people and extracts 17 COCO keypoints
2. We engineer geometric features from those keypoints
3. A trained classifier predicts injury status from those features

In [ ]:
!pip install -q ultralytics scikit-learn opencv-python-headless joblib

In [ ]:
import numpy as np
import json
import joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

np.random.seed(42)

## 1. COCO Keypoint Layout

YOLOv8-pose outputs 17 keypoints in COCO format:

| Index | Keypoint |
|-------|----------|
| 0 | Nose |
| 1 | Left Eye |
| 2 | Right Eye |
| 3 | Left Ear |
| 4 | Right Ear |
| 5 | Left Shoulder |
| 6 | Right Shoulder |
| 7 | Left Elbow |
| 8 | Right Elbow |
| 9 | Left Wrist |
| 10 | Right Wrist |
| 11 | Left Hip |
| 12 | Right Hip |
| 13 | Left Knee |
| 14 | Right Knee |
| 15 | Left Ankle |
| 16 | Right Ankle |

## 2. Synthetic Keypoint Data Generation

We generate realistic keypoint data for three pose archetypes:
- **Standing** (label 0): vertical body, head above hips above ankles
- **Lying down** (label 1): horizontal body, all keypoints at similar y-values

We add Gaussian noise and random scaling to simulate real-world variation.

In [ ]:
def generate_standing_keypoints(n_samples=300):
    """Generate synthetic keypoints for a standing person."""
    samples = []
    for _ in range(n_samples):
        cx = np.random.uniform(100, 540)
        cy = np.random.uniform(200, 400)
        scale = np.random.uniform(0.7, 1.3)
        noise = np.random.normal(0, 5, (17, 2))

        # Template: standing person (x, y) relative to center of hips
        template = np.array([
            [0, -180],   # 0  nose
            [-8, -190],  # 1  left eye
            [8, -190],   # 2  right eye
            [-15, -180], # 3  left ear
            [15, -180],  # 4  right ear
            [-30, -140], # 5  left shoulder
            [30, -140],  # 6  right shoulder
            [-40, -80],  # 7  left elbow
            [40, -80],   # 8  right elbow
            [-35, -20],  # 9  left wrist
            [35, -20],   # 10 right wrist
            [-20, 0],    # 11 left hip
            [20, 0],     # 12 right hip
            [-22, 80],   # 13 left knee
            [22, 80],    # 14 right knee
            [-22, 160],  # 15 left ankle
            [22, 160],   # 16 right ankle
        ], dtype=float)

        # Random slight lean
        lean_angle = np.random.normal(0, 0.1)  # radians, small lean
        cos_a, sin_a = np.cos(lean_angle), np.sin(lean_angle)
        rot = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
        template = template @ rot.T

        kps = template * scale + np.array([cx, cy]) + noise
        samples.append(kps)
    return np.array(samples)


def generate_lying_keypoints(n_samples=300):
    """Generate synthetic keypoints for a person lying on the ground."""
    samples = []
    for _ in range(n_samples):
        cx = np.random.uniform(100, 540)
        cy = np.random.uniform(300, 450)
        scale = np.random.uniform(0.7, 1.3)
        noise = np.random.normal(0, 6, (17, 2))

        # Lying on back or side — body is roughly horizontal
        direction = np.random.choice([-1, 1])  # facing left or right
        template = np.array([
            [direction * -180, 0],    # 0  nose
            [direction * -188, -8],   # 1  left eye
            [direction * -188, 8],    # 2  right eye
            [direction * -178, -15],  # 3  left ear
            [direction * -178, 15],   # 4  right ear
            [direction * -135, -25],  # 5  left shoulder
            [direction * -135, 25],   # 6  right shoulder
            [direction * -75, -35],   # 7  left elbow
            [direction * -75, 35],    # 8  right elbow
            [direction * -15, -30],   # 9  left wrist
            [direction * -15, 30],    # 10 right wrist
            [direction * 0, -18],     # 11 left hip
            [direction * 0, 18],      # 12 right hip
            [direction * 80, -20],    # 13 left knee
            [direction * 80, 20],     # 14 right knee
            [direction * 155, -20],   # 15 left ankle
            [direction * 155, 20],    # 16 right ankle
        ], dtype=float)

        # Random slight rotation (body not perfectly horizontal)
        tilt = np.random.normal(0, 0.15)
        cos_a, sin_a = np.cos(tilt), np.sin(tilt)
        rot = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
        template = template @ rot.T

        kps = template * scale + np.array([cx, cy]) + noise
        samples.append(kps)
    return np.array(samples)


def generate_sitting_keypoints(n_samples=150):
    """Generate sitting/crouching keypoints (label 0 = not injured)."""
    samples = []
    for _ in range(n_samples):
        cx = np.random.uniform(100, 540)
        cy = np.random.uniform(250, 420)
        scale = np.random.uniform(0.7, 1.3)
        noise = np.random.normal(0, 5, (17, 2))

        template = np.array([
            [0, -100],   # 0  nose
            [-8, -108],  # 1  left eye
            [8, -108],   # 2  right eye
            [-15, -100], # 3  left ear
            [15, -100],  # 4  right ear
            [-30, -65],  # 5  left shoulder
            [30, -65],   # 6  right shoulder
            [-45, -20],  # 7  left elbow
            [45, -20],   # 8  right elbow
            [-40, 10],   # 9  left wrist
            [40, 10],    # 10 right wrist
            [-20, 0],    # 11 left hip
            [20, 0],     # 12 right hip
            [-30, 50],   # 13 left knee (bent forward)
            [30, 50],    # 14 right knee
            [-25, 10],   # 15 left ankle (tucked back)
            [25, 10],    # 16 right ankle
        ], dtype=float)

        lean = np.random.normal(0, 0.12)
        cos_a, sin_a = np.cos(lean), np.sin(lean)
        rot = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
        template = template @ rot.T

        kps = template * scale + np.array([cx, cy]) + noise
        samples.append(kps)
    return np.array(samples)


standing = generate_standing_keypoints(400)
sitting = generate_sitting_keypoints(200)
lying = generate_lying_keypoints(400)

# Labels: 0 = OK (standing + sitting), 1 = INJURED (lying)
X_kps = np.concatenate([standing, sitting, lying], axis=0)
y = np.concatenate([
    np.zeros(len(standing) + len(sitting)),
    np.ones(len(lying))
]).astype(int)

print(f"Dataset: {len(X_kps)} samples — {(y==0).sum()} OK, {(y==1).sum()} INJURED")

## 3. Feature Engineering

We compute geometric features from raw keypoints that strongly separate standing from lying poses.

In [ ]:
def extract_features(keypoints):
    """
    Extract geometric features from a (17, 2) keypoint array.
    Returns a 1D feature vector.
    """
    kps = np.array(keypoints)
    if kps.shape == (17, 3):
        kps = kps[:, :2]

    nose = kps[0]
    l_shoulder, r_shoulder = kps[5], kps[6]
    l_hip, r_hip = kps[11], kps[12]
    l_knee, r_knee = kps[13], kps[14]
    l_ankle, r_ankle = kps[15], kps[16]

    mid_shoulder = (l_shoulder + r_shoulder) / 2
    mid_hip = (l_hip + r_hip) / 2
    mid_knee = (l_knee + r_knee) / 2
    mid_ankle = (l_ankle + r_ankle) / 2

    # Feature 1: Bounding box aspect ratio (height / width)
    x_min, y_min = kps.min(axis=0)
    x_max, y_max = kps.max(axis=0)
    bbox_w = max(x_max - x_min, 1)
    bbox_h = max(y_max - y_min, 1)
    aspect_ratio = bbox_h / bbox_w

    # Feature 2: Torso angle relative to vertical (radians)
    torso_vec = mid_hip - mid_shoulder
    torso_angle = np.arctan2(torso_vec[0], torso_vec[1])

    # Feature 3: Full body angle (nose to mid-ankle)
    body_vec = mid_ankle - nose
    body_angle = np.arctan2(body_vec[0], body_vec[1])

    # Feature 4: Vertical span / horizontal span
    span_ratio = bbox_h / bbox_w if bbox_w > 0 else 0

    # Feature 5: Nose-to-hip vertical distance (normalized by bbox height)
    nose_hip_vert = abs(nose[1] - mid_hip[1]) / bbox_h

    # Feature 6: Shoulder-to-ankle vertical distance (normalized)
    shoulder_ankle_vert = abs(mid_shoulder[1] - mid_ankle[1]) / bbox_h

    # Feature 7: Horizontal spread of all keypoints (normalized)
    horiz_spread = bbox_w / bbox_h if bbox_h > 0 else 0

    # Feature 8: Variance of y-coordinates (low = lying flat)
    y_variance = np.std(kps[:, 1]) / bbox_h if bbox_h > 0 else 0

    return np.array([
        aspect_ratio,
        abs(torso_angle),
        abs(body_angle),
        span_ratio,
        nose_hip_vert,
        shoulder_ankle_vert,
        horiz_spread,
        y_variance,
    ])


FEATURE_NAMES = [
    'aspect_ratio', 'torso_angle', 'body_angle', 'span_ratio',
    'nose_hip_vert', 'shoulder_ankle_vert', 'horiz_spread', 'y_variance'
]

X_features = np.array([extract_features(kp) for kp in X_kps])
print(f"Feature matrix shape: {X_features.shape}")
print(f"Feature names: {FEATURE_NAMES}")

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, (ax, name) in enumerate(zip(axes.flatten(), FEATURE_NAMES)):
    ax.hist(X_features[y == 0, i], bins=30, alpha=0.6, label='OK', color='green')
    ax.hist(X_features[y == 1, i], bins=30, alpha=0.6, label='INJURED', color='red')
    ax.set_title(name)
    ax.legend()
plt.tight_layout()
plt.suptitle('Feature Distributions by Class', y=1.02, fontsize=14)
plt.show()

## 4. Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

# Train a Gradient Boosting Classifier (fast + accurate)
clf = GradientBoostingClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("\n" + classification_report(y_test, y_pred, target_names=['OK', 'INJURED']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Cross-validation to confirm robustness
cv_scores = cross_val_score(clf, X_features, y, cv=5, scoring='accuracy')
print(f"5-Fold CV Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

In [ ]:
# Feature importance
importances = clf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(FEATURE_NAMES)), importances[sorted_idx], color='steelblue')
plt.xticks(range(len(FEATURE_NAMES)), [FEATURE_NAMES[i] for i in sorted_idx], rotation=45)
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

## 5. Save Model & Feature Extractor

In [ ]:
import os
os.makedirs('model', exist_ok=True)

# Save the trained classifier
joblib.dump(clf, 'model/injury_classifier.pkl')
print("Saved: model/injury_classifier.pkl")

# Save feature extraction function source so inference.py stays in sync
import inspect
with open('model/feature_config.json', 'w') as f:
    json.dump({'feature_names': FEATURE_NAMES, 'n_features': len(FEATURE_NAMES)}, f)
print("Saved: model/feature_config.json")

## 6. End-to-End Test on Real Images

Run YOLOv8-pose on a test image, extract features, and classify.

In [ ]:
# Load YOLOv8-pose (small model, fast inference)
pose_model = YOLO('yolov8s-pose.pt')
print("YOLOv8-pose model loaded.")

In [ ]:
def classify_frame(frame, pose_model, classifier):
    """
    Run full pipeline on a single frame:
    1. Detect people + keypoints
    2. Extract features
    3. Classify each person
    4. Draw results on frame
    Returns annotated frame and list of (bbox, label, confidence).
    """
    results = pose_model(frame, verbose=False)
    detections = []

    for r in results:
        if r.keypoints is None or r.boxes is None:
            continue

        keypoints_data = r.keypoints.data.cpu().numpy()
        boxes = r.boxes.xyxy.cpu().numpy()

        for kps, box in zip(keypoints_data, boxes):
            # Skip low-confidence detections
            if kps.shape[0] != 17:
                continue

            kps_xy = kps[:, :2]

            # Skip if too many keypoints are at (0,0) — not detected
            valid = np.sum(np.any(kps_xy > 0, axis=1))
            if valid < 8:
                continue

            features = extract_features(kps_xy).reshape(1, -1)
            pred = classifier.predict(features)[0]
            prob = classifier.predict_proba(features)[0]
            conf = prob[int(pred)]

            label = 'INJURED' if pred == 1 else 'OK'
            color = (0, 0, 255) if pred == 1 else (0, 200, 0)

            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

            text = f"{label} {conf:.0%}"
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
            cv2.rectangle(frame, (x1, y1 - th - 10), (x1 + tw, y1), color, -1)
            cv2.putText(frame, text, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

            # Draw keypoint skeleton
            for kp in kps_xy:
                cx, cy = int(kp[0]), int(kp[1])
                if cx > 0 and cy > 0:
                    cv2.circle(frame, (cx, cy), 4, color, -1)

            detections.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'label': label,
                'confidence': float(conf)
            })

    return frame, detections

In [ ]:
# Test with a sample image — download a test image of a standing person
!wget -q -O test_standing.jpg "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg" 2>/dev/null || echo "Download failed, create your own test image"

# Try creating a synthetic test: draw a person-like arrangement
# In practice, replace this with a real photo of someone standing or lying down
print("\n--- For best results, upload your own test images: ---")
print("1. A photo of someone standing")
print("2. A photo of someone lying on the ground")
print("\nUpload them to Colab using the file browser on the left.")

In [ ]:
# Upload your own images or use a URL — then test the full pipeline
# Replace the path below with your uploaded image

from google.colab import files
print("Upload a test image (or skip this cell if you have one already):")
try:
    uploaded = files.upload()
    test_img_path = list(uploaded.keys())[0]
except:
    test_img_path = None
    print("No file uploaded. Skipping test.")

In [ ]:
if test_img_path:
    frame = cv2.imread(test_img_path)
    if frame is not None:
        annotated, dets = classify_frame(frame.copy(), pose_model, clf)
        print(f"Detections: {dets}")
        cv2_imshow(annotated)
    else:
        print(f"Could not read image: {test_img_path}")
else:
    print("No test image available. Upload one in the cell above.")

## 7. Test with Real-World Pose Data

Run YOLOv8-pose on some real images and validate the classifier works with actual detected keypoints (not just synthetic ones).

In [ ]:
# Quick validation: generate standing and lying keypoints, run through pipeline
test_standing = generate_standing_keypoints(50)
test_lying = generate_lying_keypoints(50)

standing_preds = [clf.predict(extract_features(kp).reshape(1, -1))[0] for kp in test_standing]
lying_preds = [clf.predict(extract_features(kp).reshape(1, -1))[0] for kp in test_lying]

print(f"Standing detected as OK: {sum(1 for p in standing_preds if p == 0)}/50")
print(f"Lying detected as INJURED: {sum(1 for p in lying_preds if p == 1)}/50")

## 8. Download Model Files

Download the trained model to use in the local inference script.

In [ ]:
from google.colab import files

# Download model files
files.download('model/injury_classifier.pkl')
files.download('model/feature_config.json')
print("\nDownload the .pkl file and place it in cv_model/model/ in your project.")